# 03 · OCR Hata Analizi ve Metin Temizleme

**Vize hedefleri ile eşleşme**

| Hocanın maddesi | Bu notebook'taki bölüm |
|---|---|
| 3 · OCR'ın hangi belgelerde doğru / hatalı çalıştığını gözlemlemek | Bölüm 3–7 |
| 4 · Gereksiz boşlukları ve OCR hatalarını ele almak | Bölüm 8–9 |
| 4 · Metinleri sınıf etiketleriyle eşleştirmek, gerçek türü korumak | Bölüm 10–11 |

**Girdi:** `data/ocr_dataset.csv` (02'nin çıktısı)
**Çıktı:** `data/cleaned_dataset.csv` → sınıflandırma notebook'larının girdisi

> Önce `02_ocr_pipeline.ipynb` tamamen çalışmış olmalı.

## 1. Kurulum ve Drive

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip -q install nltk pytesseract opencv-python-headless

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
ROOT = Path("/content/drive/MyDrive/bitirme_ocr")
DATA = ROOT / "data"
FIG = ROOT / "figures"; FIG.mkdir(exist_ok=True)     # rapor görselleri
CLASSES = ["email", "form", "invoice", "resume"]

## 2. Temizleme modülü

Fonksiyonlar Drive'a `text_clean.py` olarak yazılır; Gradio demosu da aynı temizliği kullanacak.

| Fonksiyon | Ne yapar |
|---|---|
| `clean_text` | Unicode düzeltme, satır sonu tirelemesini birleştirme, `\|`, `~`, `_` gibi OCR çöplerini ve çoğu sembol olan satırları silme, fazla boşlukları kaldırma. **Büyük harf, e-posta, tutar korunur** (bilgi çıkarımında lazım) |
| `normalize_for_model` | Sınıflandırıcı için: küçük harf, e-posta → `emailaddr`, tutar → `moneyamt`, sayılar → `0` |
| `garbage_ratio` | Etiket metin olmadan OCR kalitesini tahmin eder: anlamsız token oranı |

In [ ]:
%%writefile /content/drive/MyDrive/bitirme_ocr/text_clean.py
import re
import unicodedata

# OCR'ın sık ürettiği tipografik karakterleri sadeleştir
_CHAR_MAP = str.maketrans({
    "‘": "'", "’": "'", "“": '"', "”": '"',
    "–": "-", "—": "-", "•": "-", "·": "-",
    " ": " ", "­": "",
})
_WORDISH = re.compile(r"[A-Za-z0-9]")
_NOISE_TOKEN = re.compile(r"^[^\w$%@&#+]+$")          # sadece sembol: | ~ ` ' . , _ vb.
_EDGE_JUNK_L = re.compile(r"^[|_~`'\"\s]+")
_EDGE_JUNK_R = re.compile(r"[|_~`\s]+$")
_EMAIL = re.compile(r"\b[\w.+-]+@[\w-]+(?:\.[\w-]+)+\b")
_URL = re.compile(r"\b(?:https?://|www\.)\S+", re.I)
_MONEY = re.compile(r"[$€£]\s?\d[\d,]*(?:\.\d+)?")
_NUM = re.compile(r"\d+")


def alnum_ratio(s):
    s = s.replace(" ", "")
    return sum(c.isalnum() for c in s) / len(s) if s else 0.0


def clean_text(raw, min_line_alnum=0.4):
    """OCR metnini temizler; büyük/küçük harf, e-posta, tutar gibi bilgiler korunur
    (bilgi çıkarımı aşamasında gerekecek)."""
    if not isinstance(raw, str) or not raw.strip():
        return ""
    t = unicodedata.normalize("NFKC", raw).translate(_CHAR_MAP)
    t = "".join(c for c in t if c == "\n" or c.isprintable())
    t = re.sub(r"(\w)-\n(\w)", r"\1\2", t)               # satır sonu tirelemesini birleştir
    lines = []
    for line in t.split("\n"):
        tokens = [tok for tok in line.split() if not _NOISE_TOKEN.match(tok)]
        line = " ".join(tokens)
        line = _EDGE_JUNK_R.sub("", _EDGE_JUNK_L.sub("", line))
        if len(line) < 2 or not _WORDISH.search(line):
            continue
        if alnum_ratio(line) < min_line_alnum:              # çoğu sembol olan çöp satır
            continue
        lines.append(line)
    return "\n".join(lines)


def normalize_for_model(clean):
    """Sınıflandırıcı için: küçük harf + e-posta/URL/tutar/sayıları ortak simgeye çevir.
    Böylece model 'john@acme.com' yerine 'bir e-posta adresi var' bilgisini öğrenir."""
    t = clean.lower()
    t = _EMAIL.sub(" emailaddr ", t)
    t = _URL.sub(" urladdr ", t)
    t = _MONEY.sub(" moneyamt ", t)
    t = _NUM.sub("0", t)
    t = re.sub(r"[^a-z0\s]", " ", t)
    return " ".join(t.split())


def garbage_ratio(raw):
    """Ham OCR çıktısındaki anlamsız token oranı (harf/rakam içermeyen ya da
    harf-rakam-sembol karışık kısa parçalar)."""
    toks = raw.split() if isinstance(raw, str) else []
    if not toks:
        return 1.0
    bad = 0
    for tok in toks:
        if not _WORDISH.search(tok):
            bad += 1
        elif len(tok) <= 3 and alnum_ratio(tok) < 0.67:
            bad += 1
    return bad / len(toks)


In [ ]:
import sys, importlib, json
sys.path.insert(0, str(ROOT))
import text_clean; importlib.reload(text_clean)
from text_clean import clean_text, normalize_for_model, garbage_ratio

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

df = pd.read_csv(DATA / "ocr_dataset.csv")
df["text"] = df["text"].fillna("")
print(df.shape)
df[["path", "split", "label", "n_words", "mean_conf"]].head()

## 3. OCR kalite ölçütleri

RVL-CDIP'te belgelerin doğru metni yok, bu yüzden kaliteyi **dolaylı ölçütlerle** değerlendiriyoruz:

| Ölçüt | Anlamı |
|---|---|
| `mean_conf` | Tesseract'ın kelimelere verdiği ortalama güven (0–100) |
| `n_words` | Okunan kelime sayısı (çok az ise OCR başarısız olabilir) |
| `garbage_ratio` | Anlamsız token oranı (`\|`, `~`, `l1\|` gibi) |
| `dict_ratio` | Harflerden oluşan kelimelerin İngilizce sözlükte bulunma oranı |

Kesin doğruluk (CER/WER) için 02'nin 9. bölümündeki elle yazılmış küçük doğrulama seti kullanılır.

In [ ]:
import nltk
try:
    nltk.download("words", quiet=True)
    from nltk.corpus import words as nltk_words
    VOCAB = {w.lower() for w in nltk_words.words()}
except Exception as e:
    VOCAB = None
    print("Sözlük yüklenemedi, dict_ratio atlanacak:", e)

def dict_ratio(text):
    if VOCAB is None:
        return np.nan
    toks = [t.lower() for t in text.split() if t.isalpha() and len(t) >= 3]
    return sum(t in VOCAB for t in toks) / len(toks) if toks else 0.0

df["garbage_ratio"] = df["text"].map(garbage_ratio)
df["dict_ratio"] = df["text"].map(dict_ratio)

# Gerçek veride Tesseract güveni düşük (ortalama ~40); sabit eşikler yerine veri setinin kendi dağılımı kullanılır:
# en düşük üçte birlik dilim = zayıf, en yüksek üçte birlik dilim = iyi
q_lo, q_hi = df["mean_conf"].quantile([1 / 3, 2 / 3])

def quality(r):
    if r.n_words < 5 or r.mean_conf < q_lo:
        return "zayıf"
    if r.mean_conf >= q_hi:
        return "iyi"
    return "orta"

df["ocr_quality"] = df.apply(quality, axis=1)
print(f"Kalite eşikleri (veri setinin üçte birlik dilimleri): zayıf = güven < {q_lo:.1f} veya < 5 kelime · iyi = güven ≥ {q_hi:.1f}")

## 4. Sınıf bazlı OCR kalitesi

In [ ]:
stats = df.groupby("label").agg(
    belge=("path", "count"),
    ort_kelime=("n_words", "mean"),
    ort_guven=("mean_conf", "mean"),
    cop_orani=("garbage_ratio", "mean"),
    sozluk_orani=("dict_ratio", "mean"),
).round(2)
quality_tbl = pd.crosstab(df["label"], df["ocr_quality"], normalize="index").mul(100).round(1)
quality_tbl = quality_tbl.reindex(columns=["iyi", "orta", "zayıf"], fill_value=0)
display(stats)
print("OCR kalite dağılımı (% belge):")
display(quality_tbl)
stats.join(quality_tbl).to_csv(ROOT / "ocr_kalite_tablosu.csv")   # rapora tablo olarak

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for cls in CLASSES:
    sub = df[df.label == cls]
    axes[0].hist(sub["mean_conf"], bins=30, alpha=0.5, label=cls)
    axes[1].hist(sub["garbage_ratio"], bins=30, alpha=0.5, label=cls)
axes[0].set_title("Ortalama OCR güveni"); axes[0].legend()
axes[1].set_title("Anlamsız token oranı")
quality_tbl.plot(kind="bar", stacked=True, ax=axes[2], color=["#2e7d32", "#f9a825", "#c62828"])
axes[2].set_title("OCR kalite dağılımı (%)"); axes[2].set_xlabel(""); axes[2].tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.savefig(FIG / "ocr_kalite_dagilimi.png", dpi=120); plt.show()

## 5. Ön işleme ayarlarının karşılaştırması (pilot deney)

Tüm belgelere uygulanan ayarın (v1) gerçekten iyi olup olmadığını **validation** belgelerinde kontrol ederiz.
Elle metin yazmadan karşılaştırmak için iki dolaylı ölçüt kullanılır:

| Ölçüt | Daha iyi |
|---|---|
| Ortalama OCR güveni | Yüksek |
| Sözlükte bulunan kelime oranı | Yüksek (anlamlı İngilizce kelime çok) |

Test belgeleri kullanılmaz: ayar seçimi test verisine bakılarak yapılırsa final sonuçları olduğundan iyi görünür.
Süre: ~5 dk (40 belge × 4 ayar).

In [ ]:
import cv2, time
from ocr_core import preprocess, run_ocr

VARIANTS = {
    "A · ham görüntü":               lambda im: im,
    "B · median + büyütme":          lambda im: preprocess(im, do_deskew=False, binarize=False)[0],
    "C · B + eğiklik düzeltme":      lambda im: preprocess(im, do_deskew=True, binarize=False)[0],
    "D · C + Otsu (v1, kullanılan)": lambda im: preprocess(im)[0],
}
pilot = pd.concat([g.sample(min(10, len(g)), random_state=0)
                   for _, g in df[df.split == "validation"].groupby("label")])
rows = []
t0 = time.time()
for _, r in pilot.iterrows():
    img = cv2.imread(str(DATA / r["path"]), cv2.IMREAD_GRAYSCALE)
    for name, fn in VARIANTS.items():
        o = run_ocr(fn(img))
        rows.append({"belge": r["path"], "sınıf": r["label"], "ayar": name,
                     "güven": o["mean_conf"], "kelime": o["n_words"], "sözlük_oranı": dict_ratio(o["text"])})
pilot_df = pd.DataFrame(rows)
print(f"{len(pilot)} belge × {len(VARIANTS)} ayar · {time.time() - t0:.0f} sn")

pilot_tbl = pilot_df.groupby("ayar")[["güven", "sözlük_oranı", "kelime"]].mean().round(3)
display(pilot_tbl)
print("Sınıf bazında ortalama güven:")
display(pilot_df.pivot_table(index="sınıf", columns="ayar", values="güven").round(1))
pilot_tbl.to_csv(ROOT / "pilot_on_isleme.csv")

**Nasıl yorumlanır?**

- D (v1) en iyi ya da en iyiye çok yakınsa: mevcut OCR sonuçları kalır.
- Başka bir ayar belirgin şekilde iyiyse (ör. sözlük oranında +0,05 ve üzeri): o ayarla OCR tekrar çalıştırılır.
- Sonuç ne olursa olsun bu tablo raporda **"ön işleme adımlarının OCR'a etkisi"** olarak kullanılır.

## 6. En iyi ve en kötü okunan belgeler

Her sınıftan OCR güveni **en yüksek** ve **en düşük** belgeler, OCR metninin ilk satırlarıyla birlikte gösterilir.
Bu görselleri inceleyip 6. bölümdeki gözlem tablosunu doldurun.

In [ ]:
def show_examples(cls, which="worst", n=3):
    sub = df[(df.label == cls) & (df.n_words > 0)].sort_values("mean_conf", ascending=(which == "worst")).head(n)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 9))
    for ax, (_, r) in zip(np.atleast_1d(axes), sub.iterrows()):
        ax.imshow(Image.open(DATA / r["path"]), cmap="gray"); ax.axis("off")
        snippet = "\n".join(r["text"].splitlines()[:6])[:220]
        ax.set_title(f"{cls} · güven {r.mean_conf:.0f} · {r.n_words} kelime", fontsize=11)
        ax.text(0.0, -0.02, snippet, transform=ax.transAxes, fontsize=8, va="top", family="monospace", wrap=True)
    title = "EN KÖTÜ" if which == "worst" else "EN İYİ"
    plt.suptitle(f"{title} okunan {cls} belgeleri", fontsize=14)
    plt.tight_layout(); plt.savefig(FIG / f"ocr_{which}_{cls}.png", dpi=110, bbox_inches="tight"); plt.show()

for cls in CLASSES:
    show_examples(cls, "best")
    show_examples(cls, "worst")

In [ ]:
# Neredeyse hiç metin çıkmayan belgeler (OCR tamamen başarısız)
empty = df[df.n_words < 5]
print(f"< 5 kelimelik belge: {len(empty)} ({100 * len(empty) / len(df):.1f}%)")
display(empty.groupby("label").size().rename("adet").to_frame())

In [ ]:
# Hangi faktörler OCR güveniyle ilişkili?
corr = df[["mean_conf", "n_words", "garbage_ratio", "dict_ratio", "skew_angle", "width", "height"]].corr()["mean_conf"].drop("mean_conf")
corr.round(3).to_frame("OCR güveni ile korelasyon")

## 7. Gözlemler (sizin dolduracağınız kısım)

6. bölümdeki görselleri inceleyip aşağıdaki tabloyu doldurun. Rapordaki **"OCR Hata Analizi"** bölümü buradan çıkacak.

| Sınıf | OCR'ın iyi çalıştığı durumlar | OCR'ın hatalı çalıştığı durumlar | Olası neden |
|---|---|---|---|
| email | *ör. düz daktilo metni, tek sütun* | *ör. faks başlıkları* | *ör. düşük çözünürlük, faks gürültüsü* |
| form | | *ör. kutucuklar, el yazısı alanlar* | *ör. çizgiler harf sanılıyor* |
| invoice | | *ör. tablolar, küçük punto rakamlar* | |
| resume | | | |

**Yaygın hata nedenleri için kontrol listesi:** el yazısı · tablo/çizgi · çok küçük punto · soluk baskı · eğik tarama · logo/görsel · çok sütunlu düzen · faks gürültüsü

## 8. Metin temizleme

Ham OCR metni → `text_clean` (insan okunur, bilgi çıkarımı için) → `text_norm` (model girdisi)

In [ ]:
df["text_clean"] = df["text"].map(clean_text)
df["text_norm"] = df["text_clean"].map(normalize_for_model)
df["n_words_clean"] = df["text_clean"].str.split().str.len().fillna(0).astype(int)

removed = 1 - df["n_words_clean"].sum() / max(df["n_words"].sum(), 1)
print(f"Temizlikte silinen token oranı: {removed:.1%}")
df.groupby("label")[["n_words", "n_words_clean"]].mean().round(1)

## 9. Önce / sonra örnekleri

In [ ]:
pd.set_option("display.max_colwidth", None)
for cls in CLASSES:
    pool = df[(df.label == cls) & (df.ocr_quality == "orta")]
    if pool.empty:
        pool = df[df.label == cls]
    r = pool.sample(1, random_state=7).iloc[0]
    print("=" * 90); print(f"[{cls}]  {r.path}")
    print("--- HAM OCR ---");        print(r.text[:500])
    print("--- TEMİZ ---");          print(r.text_clean[:500])
    print("--- MODEL GİRDİSİ ---");  print(r.text_norm[:300])

## 10. Etiket eşleştirme kontrolleri

Her belgenin **gerçek türünün** korunduğunu ve verinin tutarlı olduğunu doğrular.

In [ ]:
checks = {}
checks["Her satırın etiketi var"] = df["label"].notna().all()
checks["Etiketler sadece 4 sınıftan"] = set(df["label"]) <= set(CLASSES)
checks["label ↔ label_id tutarlı"] = (df["label"].map(CLASSES.index) == df["label_id"]).all()
checks["Her görüntü dosyası Drive'da"] = df["path"].map(lambda p: (DATA / p).exists()).all()
checks["Her OCR JSON dosyası Drive'da"] = df["ocr_json"].map(lambda p: (DATA / p).exists()).all()
checks["Etiket klasör adıyla aynı"] = (df["path"].str.split("/").str[2] == df["label"]).all()
checks["Belge tekrarı yok (aynı yol)"] = not df["path"].duplicated().any()
meta = pd.read_csv(DATA / "metadata.csv")
checks["01'deki tüm belgeler OCR'lanmış"] = len(df) == len(meta)
for k, v in checks.items():
    print(("✅" if v else "❌"), k)

In [ ]:
# Veri sızıntısı kontrolü: aynı metin hem train hem test'te olursa test skoru yapay olarak yükselir
nonempty = df[df.n_words_clean >= 5]
dup_groups = nonempty.groupby("text_norm").filter(lambda g: g["split"].nunique() > 1)
print(f"Farklı split'lerde aynı metne sahip belge: {len(dup_groups)}")
if len(dup_groups):
    display(dup_groups[["path", "split", "label"]].sort_values("path").head(20))

## 11. Metin sınıfları ayırt ediyor mu?

Her sınıfta en sık geçen ve sınıfı en iyi ayıran (chi-kare) kelimeler. Kelimeler sınıflar arasında belirgin farklıysa, metin tabanlı sınıflandırmanın işe yarayacağını gösterir.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import chi2

train = df[(df.split == "train") & (df.n_words_clean >= 5)]
vec = CountVectorizer(stop_words="english", token_pattern=r"(?u)\b[a-z]{3,}\b", min_df=5)
X = vec.fit_transform(train["text_norm"])
vocab = np.array(vec.get_feature_names_out())

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, cls in zip(axes, CLASSES):
    y = (train["label"] == cls).values
    scores, _ = chi2(X, y)
    freq_in = np.asarray(X[y].sum(0)).ravel(); freq_out = np.asarray(X[~y].sum(0)).ravel()
    scores = np.where(freq_in / y.sum() > freq_out / (~y).sum(), scores, 0)   # sadece bu sınıfta fazla olanlar
    top = np.argsort(scores)[::-1][:15]
    ax.barh(vocab[top][::-1], scores[top][::-1]); ax.set_title(f"{cls}: ayırt edici kelimeler")
plt.tight_layout(); plt.savefig(FIG / "ayirt_edici_kelimeler.png", dpi=120); plt.show()

## 12. Temiz veri setini kaydet

In [ ]:
df["is_empty"] = df["n_words_clean"] < 5
cols = ["path", "split", "label", "label_id", "text", "text_clean", "text_norm",
        "n_words", "n_words_clean", "mean_conf", "garbage_ratio", "dict_ratio",
        "ocr_quality", "is_empty", "ocr_json"]
df[cols].to_csv(DATA / "cleaned_dataset.csv", index=False)
print("Kaydedildi:", DATA / "cleaned_dataset.csv", df[cols].shape)
pd.crosstab(df["label"], [df["split"], df["is_empty"]])

**Boş belgeler neden silinmiyor?** `is_empty` işaretiyle tutuluyor:
- **Eğitimde** boş belgeler atlanabilir (model gürültüden öğrenmesin)
- **Testte** tutulmalı: silinirse model kolay belgelerle değerlendirilir ve başarı olduğundan yüksek görünür. Bu belgeler OCR'ın sınırını gösterir; görüntü tabanlı model (LayoutLMv3) bu durumda fark yaratabilir.

## 13. Vize özeti

Aşağıdaki hücre rapora/sunuma koyabileceğiniz sayıları tek yerde toplar.

In [ ]:
summary = {
    "Toplam belge": len(df),
    "Sınıf sayısı": df.label.nunique(),
    "Train / Val / Test": " / ".join(str((df.split == s).sum()) for s in ["train", "validation", "test"]),
    "Ortalama OCR güveni": round(df.mean_conf.mean(), 1),
    "Kalite eşikleri (güven)": f"zayıf < {q_lo:.1f} ≤ orta < {q_hi:.1f} ≤ iyi",
    "Zayıf OCR oranı en yüksek sınıf": f"{quality_tbl['zayıf'].idxmax()} (%{quality_tbl['zayıf'].max():.1f})",
    "Boş (< 5 kelime) belge": int(df.is_empty.sum()),
    "Temizlikte silinen token": f"{removed:.1%}",
    "Belge başına ort. kelime (temiz)": round(df.n_words_clean.mean(), 1),
}
pd.Series(summary).to_frame("değer")

## Çıktılar

| Dosya | Kullanım |
|---|---|
| `data/cleaned_dataset.csv` | Sınıflandırma (04) ve bilgi çıkarımı |
| `ocr_kalite_tablosu.csv` | Rapor tablosu |
| `figures/*.png` | Rapor ve sunum görselleri |
| `text_clean.py` | Demo uygulaması |

Sonraki adım → **04_baseline_tfidf_svm.ipynb**